In [1]:
!git clone https://github.com/bqmnhat/LungDiseaseDetection.git

Cloning into 'LungDiseaseDetection'...
remote: Enumerating objects: 5872, done.
remote: Total 5872 (delta 0), reused 0 (delta 0), pack-reused 5872 (from 2)
Receiving objects: 100% (5872/5872), 1.36 GiB | 37.18 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Updating files: 100% (6165/6165), done.


In [2]:
import os
data_path = os.path.join('LungDiseaseDetection', 'Data')

In [3]:
os.makedirs('/content/Data/train', exist_ok = True)
os.makedirs('/content/Data/test', exist_ok = True)
os.makedirs('/content/Data/val', exist_ok = True)

In [4]:
import shutil

def move(src_dir, dst_dir):
  for f in os.listdir(src_dir):
    shutil.move(os.path.join(src_dir, f), dst_dir)

move(os.path.join(data_path, 'train'), '/content/Data/train/')
move(os.path.join(data_path, 'test'), '/content/Data/test')
move(os.path.join(data_path, 'val'), '/content/Data/val')

In [5]:
print(len(os.listdir('/content/Data/val/PNEUMONIA')))
print(len(os.listdir('/content/Data/test/NORMAL')))

8
234


In [6]:
import numpy as np
from PIL import Image
def image_to_array(img_paths):
  img_arrays = []
  for img_path in img_paths:
    img = Image.open(img_path).convert('L').resize((128, 128))
    img_arrays.append(np.array(img))
  return np.array(img_arrays)

In [7]:
train_normal_arrays = image_to_array(os.path.join('/content/Data/train/NORMAL', f) for f in os.listdir('/content/Data/train/NORMAL'))
train_pneumonia_arrays = image_to_array(os.path.join('/content/Data/train/PNEUMONIA', f) for f in os.listdir('/content/Data/train/PNEUMONIA'))
test_normal_arrays = image_to_array(os.path.join('/content/Data/test/NORMAL', f) for f in os.listdir('/content/Data/test/NORMAL'))
test_pneumonia_arrays = image_to_array(os.path.join('/content/Data/test/PNEUMONIA', f) for f in os.listdir('/content/Data/test/PNEUMONIA'))
val_normal_arrays = image_to_array(os.path.join('/content/Data/val/NORMAL', f) for f in os.listdir('/content/Data/val/NORMAL'))
val_pneumonia_arrays = image_to_array(os.path.join('/content/Data/val/PNEUMONIA', f) for f in os.listdir('/content/Data/val/PNEUMONIA'))

In [8]:
train_normal_arrays.shape

(1575, 128, 128)

In [9]:
X_train = np.concatenate((train_normal_arrays, train_pneumonia_arrays), axis=0)
y_train = np.array([0] * len(train_normal_arrays) + [1] * len(train_pneumonia_arrays))
X_val = np.concatenate((val_normal_arrays, val_pneumonia_arrays), axis=0)
y_val = np.array([0] * len(val_normal_arrays) + [1] * len(val_pneumonia_arrays))
X_test = np.concatenate((test_normal_arrays, test_pneumonia_arrays), axis=0)
y_test = np.array([0] * len(test_normal_arrays) + [1] * len(test_pneumonia_arrays))

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

In [11]:
X_train = torch.tensor(X_train, dtype = torch.float32)
y_train = torch.tensor(y_train, dtype = torch.long)
X_val = torch.tensor(X_val, dtype = torch.float32)
y_val = torch.tensor(y_val, dtype = torch.long)
X_test = torch.tensor(X_test, dtype = torch.float32)
y_test = torch.tensor(y_test, dtype = torch.long)

In [12]:
X_train = X_train.unsqueeze(1)
X_val = X_val.unsqueeze(1)
X_test = X_test.unsqueeze(1)


print(X_train.shape)
print(X_val.shape)

torch.Size([5451, 1, 128, 128])
torch.Size([18, 1, 128, 128])


In [13]:
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

In [14]:
class LungDiseaseDetection(nn.Module):
  def __init__(self):
    super(LungDiseaseDetection, self).__init__()
    self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
    self.bn1 = nn.BatchNorm2d(32)
    self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
    self.bn2 = nn.BatchNorm2d(64)
    self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
    self.bn3 = nn.BatchNorm2d(128)
    self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
    self.bn4 = nn.BatchNorm2d(256)
    self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    self.fc1_input_features = 256 * 8 * 8
    self.fc1 = nn.Linear(self.fc1_input_features, 512)
    self.dropout = nn.Dropout(0.5)
    self.fc2 = nn.Linear(512, 2)

  def forward(self, x):
    x = self.pool(F.relu(self.bn1(self.conv1(x))))
    x = self.pool(F.relu(self.bn2(self.conv2(x))))
    x = self.pool(F.relu(self.bn3(self.conv3(x))))
    x = self.pool(F.relu(self.bn4(self.conv4(x))))
    x = x.view(-1, self.fc1_input_features)
    x = F.relu(self.fc1(x))
    x = self.dropout(x)
    x = self.fc2(x)
    return x

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LungDiseaseDetection().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [16]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [17]:
epochs = 20
best_val_loss = float('inf')
for epoch in range(epochs):
  model.train()
  total_loss = 0

  for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    optimizer.zero_grad()
    output = model(X_batch)

    loss = criterion(output, y_batch)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
  if(total_loss < best_val_loss):
    best_val_loss = total_loss
    torch.save(model.state_dict(), 'best_model.pth')
    print(f'Saved new best model at epoch {epoch} with val loss {total_loss: .4f}')
  print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader)}')

  model.eval()
  total_correct = 0
  total_samples = 0

  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      output = model(X_batch)
      _, predicted = torch.max(output, 1)

      total_samples += y_batch.size(0)
      total_correct += (predicted == y_batch).sum().item()

  accuracy = 100 * total_correct / total_samples
  print(f'Val Accuracy: {accuracy}%')

Saved new best model at epoch 0 with val loss  75.4854
Epoch 1/20, Loss: 0.4414351737856516
Val Accuracy: 88.78205128205128%
Saved new best model at epoch 1 with val loss  26.2528
Epoch 2/20, Loss: 0.15352507761260223
Val Accuracy: 91.66666666666667%
Saved new best model at epoch 2 with val loss  24.4245
Epoch 3/20, Loss: 0.14283336355890097
Val Accuracy: 89.26282051282051%
Saved new best model at epoch 3 with val loss  22.4081
Epoch 4/20, Loss: 0.13104143003764296
Val Accuracy: 86.37820512820512%
Saved new best model at epoch 4 with val loss  20.5368
Epoch 5/20, Loss: 0.1200980718536248
Val Accuracy: 84.61538461538461%
Saved new best model at epoch 5 with val loss  18.4070
Epoch 6/20, Loss: 0.1076432763822159
Val Accuracy: 94.23076923076923%
Saved new best model at epoch 6 with val loss  16.2152
Epoch 7/20, Loss: 0.09482547041704083
Val Accuracy: 94.55128205128206%
Saved new best model at epoch 7 with val loss  15.7295
Epoch 8/20, Loss: 0.09198539053792493
Val Accuracy: 83.97435897435

In [18]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
model.to(device)

LungDiseaseDetection(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=16384, out_features=512, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=512, out_features=2, bias=True)
)

In [19]:
from sklearn.metrics import confusion_matrix

with torch.no_grad():
  total_correct = 0
  total_samples = 0
  for X_batch, y_batch in val_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    output = model(X_batch)
    _, predicted = torch.max(output, 1)

    total_samples += y_batch.size(0)
    total_correct += (predicted == y_batch).sum().item()
  accuracy = 100 * total_correct / total_samples
  cm = confusion_matrix(y_batch.cpu().numpy(), predicted.cpu().numpy())
  print(f'Val Accuracy: {accuracy}%')
  print(f'Confusion Matrix:\n', cm)

Val Accuracy: 100.0%
Confusion Matrix:
 [[10  0]
 [ 0  8]]
